# Business Analytics: KPIs, Funnels, Cohorts, Retention, Churn, RFM and Domain Cases

Notebook for **Chapter 12** of the *analyst* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/analyst/ch12.html

Tip: change a number, predict the output, then run the cell again.

## 12.1 KPI and Metric Design

## 12.2 Funnels

In [ ]:
import pandas as pd
ev = pd.DataFrame({
 "user":[1,1,1,1,2,2,2,3,3,4,5,5,6],
 "step":["visit","product","cart","paid","visit","product","cart","visit","product","visit","visit","product","visit"]})
order=["visit","product","cart","paid"]
f = ev.groupby("step")["user"].nunique().reindex(order).to_frame("users")
f["step_conv_%"]=(f["users"]/f["users"].shift(1)*100).round(1)
f["overall_%"]=(f["users"]/f["users"].iloc[0]*100).round(1)
print(f)

## 12.3 Cohorts, Retention and Churn

In [ ]:
import pandas as pd, numpy as np
rng=np.random.default_rng(42)
n=1000
signup=pd.to_datetime("2025-01-01")+pd.to_timedelta(rng.integers(0,90,n),unit="D")
users=pd.DataFrame({"user_id":range(n),"signup":signup})
users["cohort"]=users.signup.dt.to_period("M")
rows=[]
for u,s in zip(users.user_id,users.signup):
    m0=s.to_period("M")
    for k in range(0,4):
        p=[1.0,0.45,0.32,0.25][k]
        if rng.random()<p: rows.append((u,m0+k))
act=pd.DataFrame(rows,columns=["user_id","month"])
act=act.merge(users[["user_id","cohort"]],on="user_id")
act["offset"]=(act.month-act.cohort).apply(lambda x:x.n)
act=act[act.month<=pd.Period("2025-06")]
tab=act.pivot_table(index="cohort",columns="offset",values="user_id",aggfunc="nunique")
print(tab)
print((tab.div(tab[0],axis=0)*100).round(1))

## 12.4 Segmentation and RFM

In [ ]:
import pandas as pd
o = pd.DataFrame({
 "cust":["A","A","A","B","B","C","D","D","D","D","E","F","G","G","H"],
 "date":pd.to_datetime(["2025-06-25","2025-06-10","2025-05-02","2025-06-20","2025-01-10","2025-02-01",
   "2025-06-28","2025-06-15","2025-06-01","2025-05-20","2025-03-05","2025-06-30","2025-04-01","2025-03-01","2025-01-05"]),
 "amount":[1200,800,500,3000,2500,400,300,350,250,400,900,150,700,600,100]})
today=pd.Timestamp("2025-07-01")
rfm=o.groupby("cust").agg(last=("date","max"),F=("date","count"),M=("amount","sum"))
rfm["R"]=(today-rfm["last"]).dt.days
rfm["R_s"]=pd.cut(rfm.R,[-1,30,90,10**6],labels=[3,2,1]).astype(int)
rfm["F_s"]=pd.cut(rfm.F,[0,1,3,10**6],labels=[1,2,3]).astype(int)
rfm["M_s"]=pd.cut(rfm.M,[0,499,1999,10**9],labels=[1,2,3]).astype(int)
def seg(r):
    if r.R_s==3 and r.F_s>=2: return "Champion"
    if r.R_s==3: return "New"
    if r.R_s==1 and r.M_s>=2: return "At risk"
    if r.R_s==1: return "Lost"
    return "Steady"
rfm["segment"]=rfm.apply(seg,axis=1)
print(rfm[["R","F","M","R_s","F_s","M_s","segment"]].to_string())
print(rfm.groupby("segment").agg(customers=("M","size"),revenue=("M","sum")).assign(share=lambda d:(d.revenue/d.revenue.sum()*100).round(1)))

## 12.5 Domain Cases: Product, Marketing, Sales, Finance, Operations

In [ ]:
import pandas as pd
d=pd.DataFrame({"user":[1,2,3,1,2,1,4,1,2,3],"day":[1,1,1,2,2,3,3,4,4,4]})
dau=d.groupby("day").user.nunique(); print(dau.to_dict())
mau=d.user.nunique(); print("avg DAU",dau.mean(),"MAU",mau,"stickiness",round(dau.mean()/mau,3))

{1: 3, 2: 2, 3: 2, 4: 3}
avg DAU 2.5 MAU 4 stickiness 0.625


In [ ]:
import pandas as pd
w=pd.DataFrame({"platform":["android","android","ios","ios","web","web"],"period":["before","after"]*3,"dau":[50000,45000,30000,29000,20000,10000]})
p=w.pivot(index="platform",columns="period",values="dau")[["before","after"]]
p["chg"]=p.after-p.before; p["pct"]=(p.chg/p.before*100).round(1)
p["share_of_drop_%"]=(p.chg/p.chg.sum()*100).round(1)
print(p); print(p[["before","after"]].sum())

period    before  after    chg   pct  share_of_drop_%
platform                                             
android    50000  45000  -5000 -10.0             31.2
ios        30000  29000  -1000  -3.3              6.2
web        20000  10000 -10000 -50.0             62.5
period
before    100000
after      84000
dtype: int64


## 12.6 Sessions, Streaks and Gaps-and-Islands

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE events(user_id INT, ts TEXT, page TEXT);
INSERT INTO events VALUES
 (1,'2025-03-01 09:00:00','home'),(1,'2025-03-01 09:12:00','product'),(1,'2025-03-01 09:30:00','cart'),
 (1,'2025-03-01 11:00:00','home'),(1,'2025-03-01 11:05:00','product'),
 (2,'2025-03-01 10:00:00','home'),(2,'2025-03-01 10:45:00','home'),(2,'2025-03-01 10:50:00','cart'),
 (3,'2025-03-02 08:00:00','home');
""")
sql = """
WITH g AS (
  SELECT user_id, ts,
         (julianday(ts) - julianday(LAG(ts) OVER (PARTITION BY user_id ORDER BY ts))) * 24 * 60 AS gap_min
  FROM events),
f AS (
  SELECT user_id, ts, gap_min,
         CASE WHEN gap_min IS NULL OR gap_min > 30 THEN 1 ELSE 0 END AS new_session
  FROM g)
SELECT user_id, ts, ROUND(gap_min, 0) AS gap_min, new_session,
       SUM(new_session) OVER (PARTITION BY user_id ORDER BY ts) AS session_no
FROM f ORDER BY user_id, ts
"""
print(pd.read_sql(sql, con).to_string(index=False))

 user_id                  ts  gap_min  new_session  session_no
       1 2025-03-01 09:00:00      NaN            1           1
       1 2025-03-01 09:12:00     12.0            0           1
       1 2025-03-01 09:30:00     18.0            0           1
       1 2025-03-01 11:00:00     90.0            1           2
       1 2025-03-01 11:05:00      5.0            0           2
       2 2025-03-01 10:00:00      NaN            1           1
       2 2025-03-01 10:45:00     45.0            1           2
       2 2025-03-01 10:50:00      5.0            0           2
       3 2025-03-02 08:00:00      NaN            1           1


In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE events(user_id INT, ts TEXT, page TEXT);
INSERT INTO events VALUES
 (1,'2025-03-01 09:00:00','home'),(1,'2025-03-01 09:12:00','product'),(1,'2025-03-01 09:30:00','cart'),
 (1,'2025-03-01 11:00:00','home'),(1,'2025-03-01 11:05:00','product'),
 (2,'2025-03-01 10:00:00','home'),(2,'2025-03-01 10:45:00','home'),(2,'2025-03-01 10:50:00','cart'),
 (3,'2025-03-02 08:00:00','home');
""")
ev = pd.read_sql("SELECT * FROM events", con, parse_dates=["ts"]).sort_values(["user_id", "ts"])
ev["gap_min"] = ev.groupby("user_id")["ts"].diff().dt.total_seconds() / 60
ev["new_session"] = (ev["gap_min"].isna() | (ev["gap_min"] > 30)).astype(int)
ev["session_no"] = ev.groupby("user_id")["new_session"].cumsum()
summary = (ev.groupby(["user_id", "session_no"])
             .agg(start=("ts", "min"), clicks=("ts", "size"), minutes=("ts", lambda s: (s.max() - s.min()).total_seconds() / 60))
             .reset_index())
print(summary.to_string(index=False))
print("sessions with 30 min limit:", len(summary))
for limit in (10, 60):
    flag = ev["gap_min"].isna() | (ev["gap_min"] > limit)
    print("sessions with", limit, "min limit:", int(flag.sum()))

 user_id  session_no               start  clicks  minutes
       1           1 2025-03-01 09:00:00       3     30.0
       1           2 2025-03-01 11:00:00       2      5.0
       2           1 2025-03-01 10:00:00       1      0.0
       2           2 2025-03-01 10:45:00       2      5.0
       3           1 2025-03-02 08:00:00       1      0.0
sessions with 30 min limit: 5
sessions with 10 min limit: 7
sessions with 60 min limit: 4


In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE activity(user_id INT, day TEXT);
INSERT INTO activity VALUES
 (1,'2025-03-01'),(1,'2025-03-02'),(1,'2025-03-03'),(1,'2025-03-05'),(1,'2025-03-06'),(1,'2025-03-06'),
 (2,'2025-03-02'),(2,'2025-03-04'),(2,'2025-03-05'),(2,'2025-03-06'),(2,'2025-03-07'),(2,'2025-03-08');
""")
sql = """WITH d AS (SELECT DISTINCT user_id, day FROM activity),
n AS (SELECT user_id, day,
             ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY day) AS rn FROM d),
k AS (SELECT user_id, day, rn, DATE(day, '-' || rn || ' day') AS grp FROM n)
SELECT * FROM k ORDER BY user_id, day"""
print(pd.read_sql(sql, con).to_string(index=False))

 user_id        day  rn        grp
       1 2025-03-01   1 2025-02-28
       1 2025-03-02   2 2025-02-28
       1 2025-03-03   3 2025-02-28
       1 2025-03-05   4 2025-03-01
       1 2025-03-06   5 2025-03-01
       2 2025-03-02   1 2025-03-01
       2 2025-03-04   2 2025-03-02
       2 2025-03-05   3 2025-03-02
       2 2025-03-06   4 2025-03-02
       2 2025-03-07   5 2025-03-02
       2 2025-03-08   6 2025-03-02


In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE activity(user_id INT, day TEXT);
INSERT INTO activity VALUES
 (1,'2025-03-01'),(1,'2025-03-02'),(1,'2025-03-03'),(1,'2025-03-05'),(1,'2025-03-06'),(1,'2025-03-06'),
 (2,'2025-03-02'),(2,'2025-03-04'),(2,'2025-03-05'),(2,'2025-03-06'),(2,'2025-03-07'),(2,'2025-03-08');
""")
sql = """WITH d AS (SELECT DISTINCT user_id, day FROM activity),
n AS (SELECT user_id, day,
             ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY day) AS rn FROM d),
k AS (SELECT user_id, day, rn, DATE(day, '-' || rn || ' day') AS grp FROM n)
SELECT user_id, MIN(day) AS start_day, MAX(day) AS end_day, COUNT(*) AS days
FROM k GROUP BY user_id, grp ORDER BY user_id, start_day"""
print(pd.read_sql(sql, con).to_string(index=False))

 user_id  start_day    end_day  days
       1 2025-03-01 2025-03-03     3
       1 2025-03-05 2025-03-06     2
       2 2025-03-02 2025-03-02     1
       2 2025-03-04 2025-03-08     5


In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE activity(user_id INT, day TEXT);
INSERT INTO activity VALUES
 (1,'2025-03-01'),(1,'2025-03-02'),(1,'2025-03-03'),(1,'2025-03-05'),(1,'2025-03-06'),(1,'2025-03-06'),
 (2,'2025-03-02'),(2,'2025-03-04'),(2,'2025-03-05'),(2,'2025-03-06'),(2,'2025-03-07'),(2,'2025-03-08');
""")
a = pd.read_sql("SELECT DISTINCT user_id, day FROM activity ORDER BY user_id, day", con)
a["day"] = pd.to_datetime(a["day"])
a["grp"] = a["day"] - pd.to_timedelta(a.groupby("user_id").cumcount() + 1, unit="D")
isl = (a.groupby(["user_id", "grp"])
         .agg(start_day=("day", "min"), end_day=("day", "max"), days=("day", "size"))
         .reset_index().drop(columns="grp"))
print(isl.to_string(index=False))
print("longest streak:", isl.groupby("user_id")["days"].max().to_dict())
isl["next_start"] = isl.groupby("user_id")["start_day"].shift(-1)
isl["gap_days"] = (isl["next_start"] - isl["end_day"]).dt.days - 1
print(isl.dropna(subset=["gap_days"])[["user_id", "end_day", "next_start", "gap_days"]].to_string(index=False))

 user_id  start_day    end_day  days
       1 2025-03-01 2025-03-03     3
       1 2025-03-05 2025-03-06     2
       2 2025-03-02 2025-03-02     1
       2 2025-03-04 2025-03-08     5
longest streak: {1: 3, 2: 5}
 user_id    end_day next_start  gap_days
       1 2025-03-03 2025-03-05       1.0
       2 2025-03-02 2025-03-04       1.0


## 12.7 Cohort Retention, Funnels and LTV in SQL, pandas and Excel

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE users(user_id INT, signup_month TEXT);
INSERT INTO users VALUES (1,'2025-01'),(2,'2025-01'),(3,'2025-01'),(4,'2025-01'),
                         (5,'2025-02'),(6,'2025-02'),(7,'2025-02'),(8,'2025-03'),(9,'2025-03');
CREATE TABLE act(user_id INT, month TEXT);
INSERT INTO act VALUES
 (1,'2025-01'),(1,'2025-02'),(1,'2025-03'),(1,'2025-04'),
 (2,'2025-01'),(2,'2025-02'),(2,'2025-04'),
 (3,'2025-01'),(3,'2025-02'),
 (4,'2025-01'),
 (5,'2025-02'),(5,'2025-03'),(5,'2025-04'),
 (6,'2025-02'),(6,'2025-03'),
 (7,'2025-02'),
 (8,'2025-03'),(8,'2025-04'),
 (9,'2025-03');
""")
sql = """
WITH a AS (
  SELECT u.signup_month AS cohort,
         (CAST(substr(x.month,1,4) AS INT)*12 + CAST(substr(x.month,6,2) AS INT))
        -(CAST(substr(u.signup_month,1,4) AS INT)*12 + CAST(substr(u.signup_month,6,2) AS INT)) AS k,
         x.user_id
  FROM act x JOIN users u USING(user_id)),
c AS (SELECT signup_month AS cohort, COUNT(*) AS size FROM users GROUP BY signup_month)
SELECT a.cohort, a.k, COUNT(DISTINCT a.user_id) AS active, c.size,
       ROUND(100.0*COUNT(DISTINCT a.user_id)/c.size, 1) AS pct
FROM a JOIN c USING(cohort) GROUP BY a.cohort, a.k ORDER BY a.cohort, a.k
"""
r = pd.read_sql(sql, con)
print(r.to_string(index=False))
print(r.pivot(index="cohort", columns="k", values="pct").to_string())

 cohort  k  active  size   pct
2025-01  0       4     4 100.0
2025-01  1       3     4  75.0
2025-01  2       1     4  25.0
2025-01  3       2     4  50.0
2025-02  0       3     3 100.0
2025-02  1       2     3  66.7
2025-02  2       1     3  33.3
2025-03  0       2     2 100.0
2025-03  1       1     2  50.0
k            0     1     2     3
cohort                          
2025-01  100.0  75.0  25.0  50.0
2025-02  100.0  66.7  33.3   NaN
2025-03  100.0  50.0   NaN   NaN


In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE users(user_id INT, signup_month TEXT);
INSERT INTO users VALUES (1,'2025-01'),(2,'2025-01'),(3,'2025-01'),(4,'2025-01'),
                         (5,'2025-02'),(6,'2025-02'),(7,'2025-02'),(8,'2025-03'),(9,'2025-03');
CREATE TABLE act(user_id INT, month TEXT);
INSERT INTO act VALUES
 (1,'2025-01'),(1,'2025-02'),(1,'2025-03'),(1,'2025-04'),
 (2,'2025-01'),(2,'2025-02'),(2,'2025-04'),
 (3,'2025-01'),(3,'2025-02'),
 (4,'2025-01'),
 (5,'2025-02'),(5,'2025-03'),(5,'2025-04'),
 (6,'2025-02'),(6,'2025-03'),
 (7,'2025-02'),
 (8,'2025-03'),(8,'2025-04'),
 (9,'2025-03');
""")
u = pd.read_sql("SELECT * FROM users", con)
a = pd.read_sql("SELECT * FROM act", con)
m = a.merge(u, on="user_id")
for c in ("month", "signup_month"):
    m[c] = pd.PeriodIndex(m[c], freq="M")
m["k"] = (m["month"] - m["signup_month"]).apply(lambda d: d.n)
active = m.groupby(["signup_month", "k"])["user_id"].nunique().unstack()
size = u.groupby("signup_month").size()
size.index = pd.PeriodIndex(size.index, freq="M")
print((active.div(size, axis=0) * 100).round(1))

k                 0     1     2     3
signup_month                         
2025-01       100.0  75.0  25.0  50.0
2025-02       100.0  66.7  33.3   NaN
2025-03       100.0  50.0   NaN   NaN


In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE events(user_id INT, ts TEXT, page TEXT);
INSERT INTO events VALUES
 (1,'2025-03-01 09:00:00','home'),(1,'2025-03-01 09:12:00','product'),(1,'2025-03-01 09:30:00','cart'),
 (1,'2025-03-01 11:00:00','home'),(1,'2025-03-01 11:05:00','product'),
 (2,'2025-03-01 10:00:00','home'),(2,'2025-03-01 10:45:00','home'),(2,'2025-03-01 10:50:00','cart'),
 (3,'2025-03-02 08:00:00','home');
""")
sql = """
WITH f AS (
  SELECT user_id,
         MIN(CASE WHEN page = 'home'    THEN ts END) AS t_home,
         MIN(CASE WHEN page = 'product' THEN ts END) AS t_product,
         MIN(CASE WHEN page = 'cart'    THEN ts END) AS t_cart
  FROM events GROUP BY user_id)
SELECT COUNT(t_home) AS home,
       COUNT(t_product) AS product,
       SUM(t_cart IS NOT NULL) AS cart_any_order,
       SUM(t_product IS NOT NULL AND t_cart > t_product) AS cart_strict
FROM f
"""
print(pd.read_sql(sql, con).to_string(index=False))

 home  product  cart_any_order  cart_strict
    3        1               2            1


In [ ]:
arpu, margin = 400, 0.30      # Rs per month, 30 percent margin
for churn in (0.05, 0.08, 0.10, 0.12):
    print("churn", churn, "LTV", round(arpu * margin / churn))
# check the formula against the long sum, and against only the first 12 months
churn = 0.08
full = sum(arpu * margin * (1 - churn) ** t for t in range(2000))
first12 = sum(arpu * margin * (1 - churn) ** t for t in range(12))
print("series:", round(full, 1), " first 12 months:", round(first12, 1))

churn 0.05 LTV 2400
churn 0.08 LTV 1500
churn 0.1 LTV 1200
churn 0.12 LTV 1000
series: 1500.0  first 12 months: 948.5


## Chapter recap

## Practice set

Problem 1 (easy). Funnel: 5,000 visits, 2,000 add to cart, 500 paid. Overall and step conversions? **Answer:** Visit to cart 40%, cart to paid 25%, overall 500 / 5000 = 10%.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/analyst/ch12.html